[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r1/01_deg_analysis.ipynb)

# R1-Q1 Week 2 — Differential Expression

You're in Week 2 of R1-Q1. The orientation notebook introduced AtGenExpress — what it is, who built it, and what's in it. This week's job is to identify, for each stress in the dataset, which genes respond to that stress. The output is one gene list per (stress, tissue) pair — the building blocks Week 3 will intersect to find the common stress core.

By the end of this notebook you will have:

- A metadata table mapping each sample (GSM) to its stress, tissue, time point, treatment status, and biological replicate.
- Walked one differential expression comparison end-to-end (cold-shoot) so every analytical choice is explicit.
- A long-format DE results table covering all eight stresses across both tissues — sixteen comparisons in total.
- Documented decisions on time-point handling, statistical thresholds, and multiple-testing correction — the material EQ#2 asks you to write up.

In [ ]:
# First we install the base package, using the package manager `pip`
# pip will pull the irilab2026 package from GitHub and install it.

!pip install git+https://github.com/geraldmc/irilab2026.git -q

## 2) Setup and data

We start by calling `setup()` from the iRI Lab library. It standardizes the
environment between Colab and a local machine — checks for GPU, mounts Drive
if you want it, sets up the cache directory used by the data loaders. You'll
see this same call in every notebook in the program; this is the last
notebook that walks through what it does.

In [ ]:
import irilab2026 as iri

iri.setup()

Now load AtGenExpress. `load_atgenexpress()` is the function we built last
week. The first time it runs it downloads the nine GEO accessions (GSE5620
through GSE5628) and parses them into pandas DataFrames; subsequent runs
read from the cache and return immediately.

In [ ]:
data = iri.load_atgenexpress()
print(list(data.keys()))

Nine keys: one matched control (`control`) plus eight abiotic stresses
(`cold`, `osmotic`, `salt`, `drought`, `genotoxic`, `uv_b`, `wounding`,
`heat`). The ninth AtGenExpress stress — oxidative — is hosted at
TAIR/NASCArrays rather than at GEO, so it is not in this loader's output.
We work with the eight that are.

Look at one of these to see the shape of the data:

In [ ]:
cold = data['cold']
print(f"Shape: {cold.shape}  (probes × samples)")
cold.iloc[:5, :3]

The rows are Affymetrix ATH1 probe IDs — about 22,810 of them, covering
most of the *Arabidopsis* genome. The columns are GEO sample IDs (those
`GSM131xxx` strings). The cell values are normalized expression
intensities.

Notice what is *not* in this matrix: the column names tell us which sample
each measurement came from, but they do not tell us *what kind of sample*
— which tissue, which time point, which biological replicate. Without that
information we cannot run differential expression. The next section
recovers it.

## 3) Parsing the metadata

Sample-level information is encoded in the *titles* of the GEO samples —
human-readable strings attached to each GSM when the dataset was deposited
in 2007. The titles look like:

    AtGen_6-0011_Control-Shoots-0h_Rep1
    AtGen_6-0712_Cold-Roots-1h_Rep2
    AtGen_6-9999_Heat-Shoots-12h_Rep1

The format is consistent: an internal experiment code, then condition,
tissue, time point, and replicate number, all separated by hyphens or
underscores. Our job is to take these titles and produce a tidy table
indexed by GSM with one column per field.

We use GEOparse — the same library `load_atgenexpress()` uses internally
— to read the cached SOFT files and pull out each sample's title. No
network round-trip needed: the SOFT files are already on disk from the
loader's first run.

In [ ]:
import GEOparse
from pathlib import Path

# The loader puts SOFT files in the iRI Lab cache. Find them all.
soft_files = sorted(iri.cache_dir().rglob('*.soft.gz'))
for p in soft_files:
    print(p)

Nine SOFT files, one per GEO series. Open one of them and look at the
sample titles to confirm the format we expect:

In [ ]:
gse = GEOparse.get_GEO(filepath=str(soft_files[1]), silent=True)  # GSE5621 = cold

for gsm_id, gsm in list(gse.gsms.items())[:6]:
    title = ' '.join(gsm.metadata.get('title', []))
    print(f"{gsm_id}  {title}")

Six cold samples. The structure is exactly as expected: stress name
(`Cold`), tissue (`Shoots` or `Roots`), time point (in hours), replicate
(`Rep1` or `Rep2`).

### 3.1) Designing the parser

The strategy is the one the Pass 1 metadata script used: parse what is
parseable, flag what is not. Guessing on ambiguous fields would silently
mislabel samples, which is worse than leaving fields blank and dealing
with the gaps explicitly.

Two of the four fields need regex. Tissue is a simple keyword check
(`shoot` or `root`); stress comes directly from the GSE the sample
belongs to (no parsing needed); time and replicate need pattern matching
because their values vary.

The time regex is worth one paragraph because there is a real subtlety in
it. AtGenExpress titles use underscores after the time unit
(`0.25h_Rep1`). Python's regex word boundary `\\b` does not trigger
before an underscore — Python treats underscore as a word character — so
a naive `\\b` after the unit fails to match. The regex below uses an
explicit lookahead that accepts end-of-string, whitespace, underscore,
and common delimiters.

In [ ]:
import re

# Time point: matches '0h', '0.25h', '0.5 h', '30 min', etc.
TIME_RE = re.compile(
    r'(\d+(?:\.\d+)?)\s*'
    r'(hours?|hrs?|h|minutes?|mins?|m)'
    r'(?=$|[\s_\-,;|/])',
    re.IGNORECASE,
)

# Replicate: matches 'Rep1', 'replicate 2', '_R1', etc.
REP_RE = re.compile(
    r'(?:replicate|rep|biological\s*replicate)\D*?(\d+)'
    r'|_R(\d+)\b',
    re.IGNORECASE,
)

def extract_time(title):
    m = TIME_RE.search(title)
    if not m:
        return None
    value = float(m.group(1))
    unit = m.group(2).lower()
    return value / 60.0 if unit.startswith('m') else value

def extract_rep(title):
    m = REP_RE.search(title)
    if not m:
        return None
    for g in m.groups():
        if g:
            return int(g)
    return None

def extract_tissue(title):
    t = title.lower()
    if 'root' in t:
        return 'root'
    if any(w in t for w in ('shoot', 'leaf', 'leaves', 'rosette', 'aerial')):
        return 'shoot'
    return None

Quick sanity check on a few representative titles before we run across
the whole dataset:

In [ ]:
test_titles = [
    'AtGen_6-0011_Control-Shoots-0h_Rep1',
    'AtGen_6-0712_Cold-Roots-1h_Rep2',
    'AtGen_6-9999_Heat-Shoots-12h_Rep1',
    'AtGen_6-9999_Drought-Roots-0.25h_Rep2',
]

for t in test_titles:
    print(t)
    print(f"  tissue={extract_tissue(t)}  time={extract_time(t)}h  rep={extract_rep(t)}")

All four fields parse cleanly on a representative spread, including the
fractional time point (0.25h) and the underscore-after-unit case that
required the lookahead.

### 3.2) Apply across all nine GSEs

Stress comes from the GSE the sample belongs to. The mapping is fixed by
the dataset's design and lives in one dict:

In [ ]:
import pandas as pd

GSE_TO_STRESS = {
    'GSE5620': 'control',
    'GSE5621': 'cold',
    'GSE5622': 'osmotic',
    'GSE5623': 'salt',
    'GSE5624': 'drought',
    'GSE5625': 'genotoxic',
    'GSE5626': 'uv_b',
    'GSE5627': 'wounding',
    'GSE5628': 'heat',
}

rows = []
for gse_id, stress in GSE_TO_STRESS.items():
    soft_path = next(p for p in soft_files if gse_id in p.name)
    gse = GEOparse.get_GEO(filepath=str(soft_path), silent=True)
    for gsm_id, gsm in gse.gsms.items():
        title = ' '.join(gsm.metadata.get('title', []))
        rows.append({
            'GSM': gsm_id,
            'GSE': gse_id,
            'stress': stress,
            'tissue': extract_tissue(title),
            'time_h': extract_time(title),
            'replicate': extract_rep(title),
            'title': title,
        })

metadata = pd.DataFrame(rows).set_index('GSM')
print(f"Metadata rows: {len(metadata)}")
metadata.head()

248 rows, one per sample. Now validate.

### 3.3) Validation 1 — chip counts

The Pass 1 reality-check pinned the per-stress chip counts against
Hahn 2013 §4.1. If our parser is right, a groupby on `stress` should
reproduce them exactly:

In [ ]:
counts = metadata.groupby('stress').size()
expected = {
    'control': 36,
    'cold': 24, 'osmotic': 24, 'salt': 24, 'genotoxic': 24,
    'drought': 28, 'uv_b': 28, 'wounding': 28,
    'heat': 32,
}

for stress, n_obs in counts.items():
    n_exp = expected.get(stress, '?')
    match = 'OK' if n_obs == n_exp else f'MISMATCH (expected {n_exp})'
    print(f"  {stress:10s}  observed={n_obs:3d}   {match}")

All counts match. That's the strongest signal we have that the parser is
not silently mislabeling — Hahn 2013's chip counts come from a different
source (the published methods) and converge on the same numbers our
parser produced from the GEO metadata.

### 3.4) Validation 2 — time grid

The Pass 1 reality-check also found a clean six-point common core —
{0.5, 1, 3, 6, 12, 24} hours — shared by all eight stresses, plus a
0h baseline that lives only in the control series, and a few
stress-specific extra time points. A pivot table makes the grid
visible:

In [ ]:
time_grid = metadata.pivot_table(
    index='stress',
    columns='time_h',
    values='GSE',
    aggfunc='count',
    fill_value=0,
)
time_grid

The grid matches the reality-check exactly:

- `control` covers all nine time points (0, 0.25, 0.5, 1, 3, 4, 6, 12, 24h).
- `cold`, `osmotic`, `salt`, `genotoxic` share the 6-point core.
- `drought`, `uv_b`, `wounding` add 0.25h.
- `heat` adds 4h.
- The 0h baseline is in `control` only.

Two analytical consequences worth flagging now, both load-bearing for the
DE design in Sections 4 and 5:

1. **Stress samples have no within-stress 0h baseline.** When we compare
   stress to control, the comparison is "stress at time *t*" vs
   "control at time *t*" — same time point, different GSE — not "stress
   at time *t*" vs "stress at 0h."

2. **The 6-point core is the unit of cross-stress comparison.** When
   Week 3 intersects DE lists across stresses, the cleanest comparison
   uses the time points that all eight stresses share.

### Validation 3 — anything still flagged?

In [ ]:
unparsed = metadata[
    metadata['tissue'].isna()
    | metadata['time_h'].isna()
    | metadata['replicate'].isna()
]
print(f"Rows with missing fields: {len(unparsed)}")
unparsed.head() if len(unparsed) else None

Zero. Every sample parsed cleanly across all four fields.

### 3.5) One more move before Section 4

We will want to compare each stress sample to its matched control sample
(same tissue, same time point, same replicate). That comparison reads
more naturally if we add a `treatment` column — `True` if the sample
came from a stress GSE, `False` if it came from the control GSE.

In [ ]:
metadata['treatment'] = metadata['stress'] != 'control'
metadata['treatment'].value_counts()

212 stress samples (8 stresses × 2 tissues × varying time points × 2
replicates), 36 controls (1 control series × 2 tissues × 9 time points
× 2 replicates). The metadata DataFrame is what Section 4 will use as
the design table for the limma fit.

## 4) What "differential expression" means here

Section 3 produced a metadata table. The job now is to find, for each stress and each tissue, which genes respond to the stress. A gene "responds" if its expression intensities in stress samples differ more than chance would predict from its expression intensities in matched controls.

That comparison sounds straightforward — a t-test per gene — but two features of this dataset make it harder than that. First, only two biological replicates per condition (n=2), which leaves a per-gene t-test with two degrees of freedom and a variance estimate too unstable to trust. Second, each stress is sampled at six time points across the response, which means the question isn't "is this gene different at *one* moment" but "does this gene respond *across* the time course." Sections 4.2 and 4.3 address each of these in turn.

### 4.1) Why limma, and a note on the Python implementation

The standard tool for differential expression on small-n microarray data is **limma** (Smyth 2004). Limma's central move is *empirical Bayes moderation* of per-gene variance estimates: rather than estimating each gene's variance from its own two replicates alone, limma shrinks the per-gene variance toward a global prior estimated across all ~22,000 genes on the array. The moderated variance for a given gene is, schematically:

$$s^2_{\text{moderated}} = \frac{d_0 \, s_0^2 + d \, s_g^2}{d_0 + d}$$

where $s_g^2$ is the per-gene variance from this gene's replicates, $s_0^2$ is the global prior variance pooled across all genes, and $d_0, d$ are degrees of freedom for the prior and the gene respectively. The effect is that genes with extreme per-gene variances (high or low) get pulled toward the global average, which stabilizes the gene rankings and makes the lists we hand to Week 3 reproducible under reasonable threshold changes.

**A note on the Python implementation:** Limma is canonical in R/Bioconductor and most published microarray analyses use the R version. We use **InMoose** (Colange et al. 2025), a peer-reviewed Python port of limma that produces numerically identical output on microarray data — verified against the R original across 12 microarray datasets, with Pearson correlations of 1.0 for p-values and log-fold-changes. InMoose lets us stay in pure Python without sacrificing the method.

### 4.2) The time-course design

AtGenExpress samples each stress at six common time points after exposure (0.5, 1, 3, 6, 12, 24 hours). We could run a separate stress-vs-control t-test at each time point and take the union — "DE at any time" — but this approach has two problems: six tests per gene inflates the false-positive rate at any fixed significance threshold, and the union step doesn't account for multiple testing across time points.

The better approach uses limma's **moderated F-test**. We build a single design matrix encoding treatment status, time point, and their interaction, then ask one question per gene: *is there any effect of stress across this time course?* One test per gene, regardless of how many time points; captures transient early-response genes (CBF-family transcription factors peaking at 1-3 hours) and sustained late-response genes (COR genes, dehydrins) in the same test. This is what `eBayes()` plus `topTable()` will produce in Section 5: per-gene F-statistic, raw p-value, adjusted p-value, and the largest log-fold-change observed across the time course.

### 4.3) What you'll have at the end of Section 5

Section 5 walks one (stress, tissue) pair — cold-shoot — end-to-end with every analytical choice visible. The output is a DataFrame indexed by gene, with columns for the moderated F-statistic, raw p-value, adjusted p-value (Benjamini-Hochberg), and the maximum log-fold-change across time points.

Section 6 surfaces the threshold choices (adjusted-p cutoff, log-fold-change minimum) that turn the F-test output into a DE gene list — those are decisions you'll make and document for EQ#2, not defaults baked into a function call. Section 7 scales the analysis to all 16 (stress, tissue) pairs and produces a long-format table covering all of them; Section 8 sanity-checks the result against what we'd expect biologically.

## 5) An end-to-end comparison

Section 4 introduced limma and the moderated F-test conceptually. This section walks one (stress, tissue) comparison — cold-shoot — end-to-end on the actual data, with every analytical step visible. Once the pattern is clear here, Section 7 will scale it to all sixteen (stress, tissue) pairs.

We pick cold-shoot because cold is the most-studied stress in this dataset and shoot tissue (above ground) is where most of the well-known cold-responsive genes operate. That means we have a strong literature-based expectation for what the top hits should look like, which makes Section 5.10's sanity check meaningful.

### 5.1) Subset to cold-shoot samples and matched controls

A single differential expression comparison needs two groups: stress samples and matched controls. For cold-shoot specifically:

- **Stress side:** cold-stressed shoot samples at the six common time points (0.5, 1, 3, 6, 12, 24 hours). With two biological replicates each, that's 12 samples.
- **Control side:** control shoot samples at the same six time points. Again 12 samples.

The "matched" part is important. We don't pool all controls together; we pair each stress sample with its time-matched control. A cold-shoot-3h sample is compared against control-shoot-3h, not against a generic "control" average. This is how the time-course design we settled on in Section 4.3 actually gets executed.

In [ ]:
# Common time points across all eight stresses (from the reality check)
COMMON_TIMEPOINTS = [0.5, 1.0, 3.0, 6.0, 12.0, 24.0]

# Cold-shoot stress samples
cold_shoot_meta = metadata[
    (metadata['stress'] == 'cold')
    & (metadata['tissue'] == 'shoot')
    & (metadata['time_h'].isin(COMMON_TIMEPOINTS))
]

# Matched controls: control samples at the same time points, same tissue
control_shoot_meta = metadata[
    (metadata['stress'] == 'control')
    & (metadata['tissue'] == 'shoot')
    & (metadata['time_h'].isin(COMMON_TIMEPOINTS))
]

# Stack them — this is the sample set for the comparison
comparison_meta = pd.concat([cold_shoot_meta, control_shoot_meta])
print(f"Cold-shoot stress samples:    {len(cold_shoot_meta)}")
print(f"Cold-shoot matched controls:  {len(control_shoot_meta)}")
print(f"Total samples in comparison:  {len(comparison_meta)}")
comparison_meta[['stress', 'tissue', 'time_h', 'treatment', 'replicate']].sort_values(['treatment', 'time_h', 'replicate'])

24 samples in the comparison — 12 stress, 12 control, paired across the six common time points with two biological replicates each. The sorted table at the bottom should show two control samples at each of the six time points, then two cold samples at each of the same six time points. If you see anything else — missing time points, extra samples, a single replicate at some point — stop and check Section 3's metadata before going further.

Now pull the expression measurements for these samples. The cold-shoot stress measurements live in `data['cold']`, and the matched controls live in `data['control']`. We extract just the columns (samples) we need from each and stack them into one matrix.

In [ ]:
# Pull the expression columns for the samples we just selected.
# data['cold'] and data['control'] are gene × sample DataFrames, so we
# select columns (GSMs) by intersecting with our comparison metadata's index.
cold_cols = [gsm for gsm in cold_shoot_meta.index if gsm in data['cold'].columns]
ctrl_cols = [gsm for gsm in control_shoot_meta.index if gsm in data['control'].columns]

expr_matrix = pd.concat(
    [data['cold'][cold_cols], data['control'][ctrl_cols]],
    axis=1,
)

print(f"Expression matrix shape: {expr_matrix.shape}  (genes × samples)")
print(f"Columns are in this order:")
for gsm in expr_matrix.columns:
    row = comparison_meta.loc[gsm]
    print(f"  {gsm}  {row['stress']:8s}  t={row['time_h']:4.1f}h  rep={int(row['replicate'])}")

The matrix has ~22,810 rows (Affymetrix ATH1 probes) and 24 columns (samples). The column order matters for the next step — InMoose's `lmFit` will pair each column of the expression matrix to the corresponding row of the design matrix we're about to build, position by position. The print-out above lets you verify that order before we proceed.

### 5.2) Build the design matrix

A design matrix tells limma *what each sample represents* in the experiment. For our time-course comparison we need three pieces of information per sample:

- **Treatment status** — is this a stress sample or a control sample?
- **Time point** — at which of the six time points was this sample taken?
- **The interaction between them** — does the stress effect *change* across time points?

The interaction term is what makes this a time-course analysis rather than six independent comparisons. It encodes "the difference between stress and control may be different at 0.5h than at 24h," which is exactly the biology we expect (CBF transcription factors peak early; downstream COR genes peak later). When we run the moderated F-test in Section 5.9, the test asks: *is there any combination of treatment and time-point-specific treatment effect that differs from zero?* — and the design matrix is what defines what "any combination" means.

We build the design matrix with `patsy`, which is the standard Python tool for converting a metadata DataFrame into a design matrix using R-style formulas.